In [1]:
import pandas as pd
import sqlite3
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

In [2]:
path = 'dataset/cleaned/domain.csv'
df = pd.read_csv(path)

path_nsw = 'dataset/cleaned/nsw.csv'
df_nsw = pd.read_csv(path_nsw)

In [3]:
conn = sqlite3.connect('database/Realestate.db')
df.to_sql("Domain", conn, if_exists="replace", index=False)
df_nsw.to_sql("NSW", conn, if_exists="replace", index=False)

1552485

### 1. Australian Property Market - Price Distribution by State

In [4]:
query_market_state = """
    SELECT 
    State,
    COUNT(*) AS total_properties,
    COUNT(DISTINCT Type) AS property_types,
    ROUND(AVG(Price), 2) AS avg_price,
    ROUND(MIN(Price), 2) AS min_price,
    ROUND(MAX(Price), 2) AS max_price
    FROM Domain
    WHERE Price IS NOT NULL
    GROUP BY State
    ORDER BY avg_price DESC;
    """
market_state = pd.read_sql_query(query_market_state, conn)
print("Property Market Analysis by State in Australia")
print(market_state)

Property Market Analysis by State in Australia
  state  total_properties  property_types   avg_price  min_price  max_price
0   nsw              7913              16  1187994.98    95000.0  7450000.0
1   qld              2856              16   876433.58    95000.0  7450000.0
2   vic              3682              16   792265.32    99000.0  6800000.0
3   act               448              14   726397.02   180000.0  2439000.0
4    wa               862              16   674629.77    95000.0  3980000.0
5    sa               401              16   596833.28   100000.0  4392145.0
6   tas               220              15   537844.24   100000.0  3650000.0
7    nt                55              11   434586.45   140000.0  1110000.0


### 2. Statistical Summary of Property Prices by Type in NSW

In [5]:
query_type_nsw = """
    SELECT 
    Type, 
    COUNT(*) AS total_listings,
    ROUND(AVG(Price), 0) AS avg_price,
    ROUND(MIN(Price), 0) AS min_price,
    ROUND(MAX(Price), 0) AS max_price
    FROM Domain
    WHERE State = 'nsw'
    GROUP BY Type
    ORDER BY total_listings DESC;
    """
type_nsw = pd.read_sql_query(query_type_nsw, conn)
print("Property Type Analysis in New South Wales (NSW)")
print(type_nsw)

Property Type Analysis in New South Wales (NSW)
                             Type  total_listings  avg_price  min_price  max_price
0                   Semi-detached             991  1609251.0   330000.0  7000000.0
1                         Terrace             827  2212212.0   300000.0  7450000.0
2                          Studio             724   496031.0   102500.0  2200000.0
3                           Villa             714   868266.0   165000.0  2285000.0
4         Apartment / Unit / Flat             649   905178.0   196000.0  4465000.0
5                          Duplex             584  1388329.0   370000.0  4500000.0
6   New apartments / off the plan             563  1169479.0   365000.0  7100000.0
7              New house and land             473  1219694.0   330000.0  7100000.0
8                       Townhouse             450  1162447.0   412500.0  5750000.0
9                     Vacant land             445   682164.0   100000.0  4950000.0
10              Retirement Living      

### 3. Impact of Property Features on Price - Beds, Baths, and Parking

In [6]:
query_property_config = """
SELECT Beds, Baths, Parking, AVG(Price) AS avg_price
FROM Domain
GROUP BY Beds, Baths, Parking
ORDER BY avg_price DESC
LIMIT 10
"""
property_config = pd.read_sql_query(query_property_config, conn)
print("Ranking of property types by average value")
print(property_config)

Ranking of property types by average value
   Beds  Baths  Parking     avg_price
0  10.0    6.0      3.0  4.225000e+06
1  10.0    0.0      0.0  3.960000e+06
2   4.0    5.0      4.0  3.750000e+06
3   8.0    3.0      1.0  3.750000e+06
4   2.0    3.0      0.0  3.725000e+06
5   4.0    4.0      5.0  3.677500e+06
6   5.0    0.0      2.0  3.650000e+06
7   6.0    6.0      0.0  3.650000e+06
8   7.0    3.0      4.0  3.468000e+06
9   4.0    3.0      0.0  3.442857e+06


### 4. Local Price Concentration and Market Value Distribution in NSW

In [7]:
query_suburb_sales = """
SELECT 
    "Property locality",
    COUNT(*) AS total_sales,
    ROUND(AVG("Purchase price"), 0) AS avg_price,
    ROUND(SUM("Purchase price"), 0) AS total_value,
    ROUND(MIN("Purchase price"), 0) AS min_price,
    ROUND(MAX("Purchase price"), 0) AS max_price
FROM NSW
WHERE "Property locality" IS NOT NULL
GROUP BY "Property locality"
HAVING total_sales > 5
ORDER BY avg_price DESC
LIMIT 10;
"""

suburb_sales = pd.read_sql_query(query_suburb_sales, conn)
print("Top Suburbs by Average and Total Sales (NSW)")
print(suburb_sales)

Top Suburbs by Average and Total Sales (NSW)
  Property locality  total_sales  avg_price   total_value  min_price   max_price
0         bradfield           20  7729350.0  1.545870e+08  1375000.0  12000000.0
1    badgerys creek           20  7042619.0  1.408524e+08  1623700.0  11750000.0
2     duffys forest           62  5672624.0  3.517027e+08     2704.0  11314000.0
3       longueville          198  5106532.0  1.011093e+09   600000.0  13925000.0
4          clontarf          201  4623580.0  9.293395e+08    52000.0  11200000.0
5       dawes point          164  4608003.0  7.557124e+08    20000.0  14650000.0
6      linley point           35  4545068.0  1.590774e+08    43500.0   9980000.0
7       whale beach           85  4517718.0  3.840060e+08    60000.0  10680000.0
8    huntleys point           10  4336500.0  4.336500e+07   290000.0   9800000.0
9         northwood          106  4235041.0  4.489144e+08    10000.0  11200000.0


### 5. Yearly Peak Month Analysis of Property Prices in NSW (From 2000)

In [8]:
query_year_analysis = """
WITH monthly_stats AS (
    SELECT 
        strftime('%Y', "Contract date") AS year,
        strftime('%Y-%m', "Contract date") AS month,
        COUNT(*) AS total_sales,
        ROUND(AVG("Purchase price"), 0) AS avg_price,
        ROUND(MIN("Purchase price"), 0) AS min_price,
        ROUND(MAX("Purchase price"), 0) AS max_price
    FROM NSW
    WHERE "Contract date" IS NOT NULL
      AND CAST(strftime('%Y', "Contract date") AS INTEGER) > 1999
    GROUP BY year, month
)
SELECT m1.*
FROM monthly_stats m1
WHERE m1.avg_price = (
    SELECT MAX(m2.avg_price)
    FROM monthly_stats m2
    WHERE m2.year = m1.year
)
ORDER BY m1.year ASC;
"""

year_analysis = pd.read_sql_query(query_year_analysis, conn)
print("Yearly Peak Month Analysis of Property Prices in NSW (From 2000)")
print(year_analysis)

Yearly Peak Month Analysis of Property Prices in NSW (From 2000)
    year    month  total_sales  avg_price  min_price   max_price
0   2000  2000-05            3  2222000.0   416000.0   5000000.0
1   2001  2001-11            5  1823240.0   425000.0   4000000.0
2   2002  2002-10            9  1250792.0     2732.0   4150000.0
3   2003  2003-01            1  1400000.0  1400000.0   1400000.0
4   2004  2004-12            2  2174000.0   948000.0   3400000.0
5   2005  2005-07            6  1052619.0    58000.0   3477000.0
6   2006  2006-11            7  2398857.0     7000.0   6900000.0
7   2007  2007-11            8   985625.0    45000.0   4000000.0
8   2008  2008-03            4  1385000.0   462000.0   3600000.0
9   2009  2009-11           14  1397324.0   360000.0   4000000.0
10  2010  2010-07           16  2018641.0    15053.0   9140000.0
11  2011  2011-11           27   981712.0     8950.0   7805000.0
12  2012  2012-04           39   914586.0    90000.0   5000000.0
13  2013  2013-12        